1. DE do not only work/optimize pipelines, sometimes they can change the design of upstream architect or resolve problems from the source 

2. Aggregate fact and turn the minto dimensions

# What Is Facts?
- Think of fact as something that happened or occurered. For example, 
  - a user logs in to an app
  - a transaction is made
  - you run a mile with your fitbit

    ![](./images/2/What_Is_Fact_Example.png)
    
- It should be **_atomic_** - lowest granularity
- It **_cannot be changed_**

## Challenges
- **_Amount of data_** x10 or x100 compared to dimension
- Need **_more context_** for analysis
  - Conversion funnel example

    ![](./images/2/Conersion_Funnel_Example.png)

- **_Duplicates_** more in fact than dimension
  - Bugs
    - e.g. App pushes logs: 1 click generates 2 logs --> Data Quality issue
  - Genuin duplicates (can be handled by Deduplication)
    - e.g. A notifications clicked now and clicked 3 hours later
      - Tracking 2 clicks in different timeframe --> Deduplication
      - If not dedup, metric looks weird (200% CTR)

# How Does Fact Modeling work?

## Fact Data vs. Raw Logs
They are **_NOT_** the same thing, but they are **_interconnected_**
- **Raw logs:**
  - **_Ugly schemas_** designed for online systems
  - Potentially **_contains duplicates_** and other quality errors
  - Usually **_short retention_**
- **Fact Data:**
  - **_Nice schemas/column names_** for analysis
  - **_Quality guarantees_** like uniqueness, not null, etc.
  - **_Longer retention_**

## Normalization vs. Denormalization

Both normlization and denormalization can cause issues.

--> The smaller the scale, the better normalization will be the option

![](./images/2/Normalization_Denormalization.png)

**Example:** Network Logs Traffic in Netflix

![](./images/2/Network_Logs_Netflix_Example.png)

Then, Fact data include App Name as app identifier

## Think as Who, Where, How, What and When? 
1. **Who:** = who is part of the event?
    - Usually are pushed out as **_IDs_**
    - Example, the user clicked the button, we hold **_user_id_**, not the entire user object
2. **Where:** = where is the event took place?
    - Similar like **_Who_** with IDs to join, but more likely to bring in additional dimensions, especially if the are **_high cardinality_**
    - Example, on app or webpage or what button clicked, location (Country, City)
3. **How:** = how is part of the event?
    - Similar to **_Where_** and **_Who_**
    - Example, he used an iphone to make this click, 
      - where: **_device_id_**
      - how: methodology that lead to click
4. **What:** = what event that happened?
    - Consider the **_proper atomicity_** for aggreations
    - Example, in notification world, Generated -> Sent -> Clicked -> onverted into Purchase -> Delivered
5. **When:** = when is the event took place?
    - Ensure to convert to the **_same timezone_**
    - Example, **_event_timestamp_** or **_event_date_**

## Other Considerations
- Fact datasets should have **_quality guarantees_**
  - No duplicates
  - All fields are available for analysis, e.g. no NULL for Who, What and When
- Fact data should generally be **_smaller_** than raw logs
  - For example, no need infor to investigate the technical issues
- Fact data should **_parse out hard-to-understand_** columns
  - String, but it sometimes is in **complex datatype**

# How Does Logging Fit into Fact Data?
- Logging brings in all the **_critical context_** for dact data
  - Usually done in collaboration with only system engineers
- Only **_necessary log_**
- **Conformance:** logging needs to be in some type of contract, shared schema/vision
  - Thrift = shared schema between apps
  
  ![](./images/2/Thrift_Schema_Example.png)

# Working with High Volume Fact Data
- **Sampling:**
  - Works best for metric-driven user cases when imprecision is not an issue
  - Does **_not_** work for all use cases
- **Bucketing:**
  - Fact data can be bucketed by on of the important dimensions (usually user)
  - Bucket joins can be much faster than shuffle joins
  - Sorted-merge bucket (SBM) joins can do joins without Shuffle at all

# Retention
- High volumes make fact data much more costly to hold onto for a long time
- Big tech had interesting approach:
  - Any fact tables **< 10 TB**, retention did not matter
    - Anonymization of facts usually happened after 60-90 days though and the data would be moved to a new table with PII stripped
  - Any fact tables **> 100TB**, very short retention (~14 days or less)

# Deduplication of Fact Data
- Fact can often be **_duplicated_**
- How do you pick the **_right window_** for deduplication? = **_time frame_** for deduplicating
  - No duplicates in a day, hour, week?
  - Look at distributions of duplicates is a good idea
- Intraday deduplicating options:
  - Streaming
  - Microbatch

## Streaming to Deduplicate
Steaming allows to capture most duplicates in a very efficient manner (immediate data consistency)
  - Windowing matters here
  - Entire day duplicates can be harder for streaming because it needs to hold onto such a **_big window of memory_**
  - A large memory of duplicates usually happen within a short time of first event
  - **Sweet spot:** **_15 mins_** to **_hourly_** windows

## Hourly Microbatch to Deduplicate
- Used to reduce landing time of daily tables that dedupe slowly
- For example, Facebook: deduplicate 50 billion notification events every day. Reduced laning time from 9 hours to 1 hour

**Step 1:**
- Dedupe **_each hour_** with GROUP BY
- Use SUM and COUNT to agrreate duplicates
- Use COLLECT_LIST to colect metadata about the duplicates

**Step 2:**
- Dedupe between hours with FULL OUTER JOIN like branches of tree
- Use left.value + right.value to keep duplicates aggregation correctly counting or CONCAT to build a continuous list

![](./images/2/Microbatch_Deduplicate_Example.png)

# ----------------Lab 1----------------

# Fact or Dimension?
They are **_blurry_** and they can be based on each other
- For example, did a user log in today?
  - **dim_is_active**: the log in event would be a fact that informs the "dim_is_active" dimension, e.g. make actions within 3 mins, did the user still active **--> activity driven = Fact**
  - **dim_is_activated**: it is state-drive, not activity driven, e.g. the acount is activated or deactivated? = 1 attribute of object user **--> Dimension**
- You can aggreate facts and turn them into dimensions
  - Is this person a "high engager" or "low engager"?
  - CASE WHEN to bucketize aggregated facts can be very useful to reduce cardinality

# Properties of Facts and Dimensions
1. **Facts:**
    - Usually **_aggregated_** when doing analytics like **_`SUM, AVG, COUNT`_**
    - Almost always **_higher volume_** than dimensions, although some fact sources are low-volume
    - Generally come from **_events_** and **_logs_**

2. **Dimensions:**
    - Usually show up in **_`GROUP BY`_** when doing analytics
    - Can be "**_high_** cardinality" or "**_low_** cardinality". For example,
      - user_id = high cardinality
      - country_id = medium cardinality
      - gender = low cardinality
    - Generally com from a **_snapshot of state_**



**Keep in mind:**
- We can aggregate **_Facts_** and turn them into **_Dimensions_**
- **Change Data Capture (CDC):** sit the in the **_blurry line_** between of fact and dimension. For example, 
  - (1) we can model a **_state change_** of a dimension as an **_event_** or as a **fact**
  - (2) we can recreate Dimension at any moment in time based on the stack of changes that happened

**Example of Airbnb:** Is the price of a night a fact or a dimension?
- Situation:
  - The host can set the price which sounds like an event
  - It can easily be SUM, AVG, COUNT like regular facts
  - Prices on Airbnb are doubles, therefore extrmely high cardinality
- Consider as:
  - Fact: When the host **_changes the setting_** that **_impacted the price_**
  - Dimension: Price being **_derived from settings_** is a dimension
  - Think as:
    - a fact has to be **_logged_**
    - a dimension comes from the **_state of things_**

# Boolean / Existence-based Fact and Dimensions
- dim_is_active, dim_bought_something, etc
  - They are usually on daily/hourly grain
- dime_has_ever_booked, dim_ever_active, dime_ever_labbled_Fake
  - These "ever" dimensions look to see if there has "ever" been a log and once it flips on way, it never goes back
  - Interesting, simple and powerful features for ML
    - Airbnb host iwth active listings who has never been booked
- "Days since" dimensions (e.g. days_since_last_active, days_since_signup, etc.)
  - Very common in Retention analytics partterns
  - Look up **_J curves_** for more details